In [ ]:
!pip install monai

In [ ]:
import os
import re
import glob
from collections import defaultdict
import torch
import numpy as np
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# MONAI Imports
from monai.config import print_config
from monai.utils import set_determinism
from monai.data import list_data_collate
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    RandFlipd,
    MapLabelValued,
    ConcatItemsd,
    DeleteItemsd,
    RandShiftIntensityd,
)
from monai.networks.nets import SegResNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.data import Dataset, decollate_batch

In [ ]:
# --- 1. CONFIGURATION ---
DATA_ROOT = '/kaggle/input/instant-odc-ai-hackathon/Train' 
MAX_EPOCHS = 100
VAL_INTERVAL = 5
BATCH_SIZE = 1
LR = 1e-4
ROI_SIZE = (128, 128, 128)

set_determinism(seed=0)

In [ ]:
# --- 2. DATA PREPARATION ---
class BraTSDataOrganizer:
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.patient_map = defaultdict(dict)
        self._scan_files()

    def _scan_files(self):
        print(f"Scanning {self.root_dir}...")
        regex_brats = re.compile(r"(BraTS2021_\d{5})")
        
        for root, _, files in os.walk(self.root_dir):
            for filename in files:
                if filename.endswith(('.nii', '.nii.gz')):
                    full_path = os.path.join(root, filename)
                    lower_name = filename.lower()
                    
                    key_type = None
                    if 'flair' in lower_name: key_type = 'flair'
                    elif 't1ce' in lower_name: key_type = 't1ce'
                    elif 't1' in lower_name: key_type = 't1' 
                    elif 't2' in lower_name: key_type = 't2'
                    elif 'seg' in lower_name: key_type = 'seg'
                    
                    if key_type:
                        match = regex_brats.search(full_path)
                        if match:
                            pid = match.group(1)
                            if os.path.getsize(full_path) > 0:
                                self.patient_map[pid][key_type] = full_path

    def get_data_list(self):
        data_list = []
        for pid, paths in self.patient_map.items():
            if all(k in paths for k in ['flair', 't1ce', 't1', 't2', 'seg']):
                data_list.append({
                    'image_flair': paths['flair'],
                    'image_t1ce': paths['t1ce'],
                    'image_t1': paths['t1'],
                    'image_t2': paths['t2'],
                    # Fallback list for simple loading if needed, though we use specific keys below
                    'image': [paths['flair'], paths['t1ce'], paths['t1'], paths['t2']], 
                    'label': paths['seg'],
                    'id': pid
                })
        print(f"Found {len(data_list)} complete cases.")
        return data_list

In [ ]:
organizer = BraTSDataOrganizer(DATA_ROOT)
all_data = organizer.get_data_list()
split_idx = int(len(all_data) * 0.8)
train_files = all_data[:split_idx]
val_files = all_data[split_idx:]

In [ ]:
# --- 3. TRANSFORMS ---
train_transforms = Compose([
    LoadImaged(keys=["image_flair", "image_t1ce", "image_t1", "image_t2", "label"]),
    EnsureChannelFirstd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2", "label"]),
    
    # --- FIX STARTS HERE ---
    # BraTS labels are 0, 1, 2, 4. We must map 4 -> 3 to make them consecutive (0, 1, 2, 3).
    # You had 1->2 before; assuming you want standard multiclass, use this:
    MapLabelValued(keys=["label"], orig_labels=[4], target_labels=[3]),
    # --- FIX ENDS HERE ---
    
    ConcatItemsd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2"], name="image", dim=0),
    DeleteItemsd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=ROI_SIZE,
        pos=1,
        neg=1,
        num_samples=1, 
        image_key="image",
        image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
])

val_transforms = Compose([
    LoadImaged(keys=["image_flair", "image_t1ce", "image_t1", "image_t2", "label"]),
    EnsureChannelFirstd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2", "label"]),
    
    # --- FIX: Apply the same mapping to validation ---
    MapLabelValued(keys=["label"], orig_labels=[4], target_labels=[3]),
    
    ConcatItemsd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2"], name="image", dim=0),
    DeleteItemsd(keys=["image_flair", "image_t1ce", "image_t1", "image_t2"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
])

In [ ]:
# --- 4. DATA LOADERS ---
train_ds = Dataset(data=train_files, transform=train_transforms)

# FIX: Added collate_fn=list_data_collate
train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=2, 
    collate_fn=list_data_collate 
)

val_ds = Dataset(data=val_files, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

# --- 5. MODEL & TRAINING SETUP ---
device = torch.device("cuda:0")

model = SegResNet(
    blocks_down=[1, 2, 2, 4],
    blocks_up=[1, 1, 1],
    init_filters=16,
    in_channels=4,
    out_channels=4, 
    dropout_prob=0.2,
).to(device)

loss_function = DiceLoss(
    smooth_nr=0, 
    smooth_dr=1e-5, 
    squared_pred=True, 
    to_onehot_y=True, 
    softmax=True
)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
# Fixed deprecated warning for newer pytorch
scaler = torch.amp.GradScaler('cuda')

In [ ]:
# --- 6. TRAINING LOOP ---
print("Starting Training...")
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0
    step = 0
    
    # WRAPPER: Create a progress bar for the current epoch
    # 'desc' sets the text description (e.g., "Epoch 1/100")
    batch_iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}", unit="batch")
    
    for batch_data in batch_iterator:
        step += 1
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
        # TQDM UPDATE: Update the progress bar with the current loss
        batch_iterator.set_postfix({"loss": f"{loss.item():.4f}"})
        
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    # We use batch_iterator.write to print without breaking the progress bar layout
    batch_iterator.write(f"Epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # VALIDATION STEP
    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)
        
        with torch.no_grad():
            # You can also wrap validation loader if you want to see validation progress
            for val_data in tqdm(val_loader, desc="Validating", leave=False):
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)
                
                val_outputs = sliding_window_inference(
                    inputs=val_inputs, 
                    roi_size=ROI_SIZE, 
                    sw_batch_size=4, 
                    predictor=model,
                    overlap=0.5,
                )
                
                val_outputs_indices = torch.argmax(val_outputs, dim=1)
                val_outputs_onehot = torch.nn.functional.one_hot(
                    val_outputs_indices.long(), num_classes=4
                ).permute(0, 4, 1, 2, 3)
                
                val_labels_indices = val_labels[:, 0,...]
                val_labels_onehot = torch.nn.functional.one_hot(
                    val_labels_indices.long(), num_classes=4
                ).permute(0, 4, 1, 2, 3)
                
                dice_metric(y_pred=val_outputs_onehot, y=val_labels_onehot)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), "best_metric_model.pth")
                batch_iterator.write(">>> Saved new best model!")
            
            batch_iterator.write(f"Validation Mean Dice: {metric:.4f}")
            batch_iterator.write(f"Best Dice: {best_metric:.4f} at Epoch {best_metric_epoch}")

print("Training Completed.")